In [ ]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load dataset
df = pd.read_csv("/content/expertise_risk_classification_dataset_rebuilt.csv")

print("Shape:", df.shape)
print()
print("Expertise distribution:")
print(df['expertise'].value_counts())
print()
print("Risk distribution:")
print(df['risk'].value_counts())
print()
print("Null values:", df.isnull().sum().to_dict())

## Train expertise model (with class_weight='balanced' to fix imbalance)

In [ ]:
x = df['text']
y_expertise = df['expertise']

x_train, x_test, y_train, y_test = train_test_split(
    x, y_expertise, test_size=0.2, random_state=42, stratify=y_expertise
)

# class_weight='balanced' fixes the imbalance between Not an Incident (70 rows)
# and Mental Health (2410 rows) — without this, the model ignores rare classes
expertise_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0))
])

expertise_model.fit(x_train, y_train)
print("Expertise model trained.")

In [ ]:
# Evaluate expertise model
expertise_preds = expertise_model.predict(x_test)
print(classification_report(y_test, expertise_preds))

In [ ]:
# Confusion matrix for expertise
cm = confusion_matrix(y_test, expertise_preds, labels=expertise_model.classes_)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=expertise_model.classes_,
            yticklabels=expertise_model.classes_)
plt.title('Expertise model — confusion matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Train risk model

In [ ]:
y_risk = df['risk']

x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(
    x, y_risk, test_size=0.2, random_state=42, stratify=y_risk
)

risk_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0))
])

risk_model.fit(x_train_r, y_train_r)
print("Risk model trained.")

In [ ]:
# Evaluate risk model
risk_preds = risk_model.predict(x_test_r)
print(classification_report(y_test_r, risk_preds))

## Sanity-check: test Not an Incident inputs

In [ ]:
test_inputs = [
    # should all predict Not an Incident / None
    "I can make it in life",
    "I can make it in life and be successful",
    "Hello, how are you?",
    "Testing 1 2 3",
    "I am proud of how far I have come",
    # should predict real categories
    "I want to commit suicide and I have no one to talk to",
    "I cannot afford my school fees and I may be deregistered",
    "I feel so stressed and overwhelmed I cannot sleep",
]

print(f"{'Input':<55} {'Expertise':<22} {'Risk'}")
print('-' * 90)
for t in test_inputs:
    exp  = expertise_model.predict([t])[0]
    risk = risk_model.predict([t])[0]
    conf = max(expertise_model.predict_proba([t])[0])
    print(f"{t[:54]:<55} {exp:<22} {risk}  (conf: {conf:.2f})")

## Save both models

In [ ]:
joblib.dump(expertise_model, 'expertise_model.pkl')
joblib.dump(risk_model, 'risk_model.pkl')
print('expertise_model.pkl saved')
print('risk_model.pkl saved')